# Hard Thresholding final fine search on Fritz

This notebook evaluates ranks 12, 14, and 18 for 300 iterations and compares them with the already computed rank-16 trajectory from the long search.

In [ ]:
import sys
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse.linalg import ArpackNoConvergence
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from hard_thresholding_hpc import (
    hard_thresholding_model_matrix_free,
    predict_pairs,
)

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ml-latest"
LONG_RESULTS_DIR = PROJECT_ROOT / "results" / "hard_thresholding_long"
RESULTS_DIR = PROJECT_ROOT / "results" / "hard_thresholding_final_fine"
HISTORIES_DIR = RESULTS_DIR / "histories"
HISTORIES_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(PROCESSED_DIR / "train_ratings.csv")
user_map = pd.read_csv(PROCESSED_DIR / "user_map.csv")
movie_map = pd.read_csv(PROCESSED_DIR / "movie_map.csv")

n_users = len(user_map)
n_movies = len(movie_map)

train_fit_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

val_user_idx = val_df["user_idx"].to_numpy(dtype=np.int64)
val_movie_idx = val_df["movie_idx"].to_numpy(dtype=np.int64)
val_ratings = val_df["rating"].to_numpy(dtype=np.float64)

print("Train fit shape:", train_fit_df.shape)
print("Validation shape:", val_df.shape)
print("Number of users:", n_users)
print("Number of movies:", n_movies)

In [ ]:
baseline_history_path = LONG_RESULTS_DIR / "selected_history.csv"
baseline_parameters_path = LONG_RESULTS_DIR / "best_parameters.csv"

if not baseline_history_path.exists() or not baseline_parameters_path.exists():
    raise FileNotFoundError(
        "Run the long-search Slurm job before the final fine search."
    )

baseline_history = pd.read_csv(baseline_history_path)
baseline_parameters = pd.read_csv(baseline_parameters_path)
baseline_rank = int(baseline_parameters.iloc[0]["rank"])
baseline_max_iter = int(baseline_history["iteration"].max())

if baseline_rank != 16 or baseline_max_iter < 300:
    raise ValueError(
        "Expected the saved baseline to be rank 16 with at least 300 iterations, "
        f"got rank={baseline_rank}, max_iter={baseline_max_iter}."
    )

print("Saved baseline parameters:")
display(baseline_parameters)

In [ ]:
def hard_validation_metrics(U, singular_values, V):
    predictions = predict_pairs(
        U,
        singular_values,
        V,
        val_user_idx,
        val_movie_idx,
    )
    predictions = np.clip(predictions, 0.5, 5.0)
    return {
        "val_rmse": float(np.sqrt(np.mean((val_ratings - predictions) ** 2))),
    }

## Final fine rank grid

Only ranks 12, 14, and 18 are trained. Rank 16 is reused from the saved 300-iteration long-search trajectory.

In [ ]:
fine_rank_values = [12, 14, 18]
fine_max_iter = 300
hard_svd_tol = 1e-4
hard_svd_maxiter = 300

print("New fine-search ranks:", fine_rank_values)
print("Saved baseline rank:", baseline_rank)
print("Iterations per new rank:", fine_max_iter)

## Final fine search

In [ ]:
# Compile the Numba prediction kernel before the timed search.
predict_pairs(
    np.ones((2, 1)),
    np.ones(1),
    np.ones((2, 1)),
    np.array([0], dtype=np.int64),
    np.array([0], dtype=np.int64),
)

fine_histories = []
fine_failed_configs = []
fine_search_start = perf_counter()

for config_number, rank in enumerate(fine_rank_values, start=1):
    history_path = HISTORIES_DIR / f"rank-{rank}.csv"
    print(f"[{config_number}/{len(fine_rank_values)}] rank={rank}")

    if history_path.exists():
        print("  using saved history")
        fine_histories.append(pd.read_csv(history_path))
        continue

    config_start = perf_counter()
    try:
        _, _, _, history_df = hard_thresholding_model_matrix_free(
            train_fit_df,
            n_users=n_users,
            n_movies=n_movies,
            rank=rank,
            n_iter=fine_max_iter,
            metrics_fn=hard_validation_metrics,
            svd_tol=hard_svd_tol,
            svd_maxiter=hard_svd_maxiter,
            random_state=42,
        )
    except ArpackNoConvergence as error:
        fine_failed_configs.append({
            "rank": rank,
            "error": repr(error),
        })
        print("  failed to converge:", repr(error))
        continue

    config_seconds = perf_counter() - config_start
    history_df["config_seconds"] = config_seconds
    history_df.to_csv(history_path, index=False)
    fine_histories.append(history_df)
    best_row = history_df.loc[history_df["val_rmse"].idxmin()]
    print(
        f"  {config_seconds / 60:.1f} min; "
        f"best iteration={int(best_row['iteration'])}; "
        f"best val RMSE={best_row['val_rmse']:.6f}"
    )

if not fine_histories:
    raise RuntimeError("Every final fine-search configuration failed")

fine_results_df = pd.concat(fine_histories, ignore_index=True)
fine_best_by_rank = (
    fine_results_df.loc[
        fine_results_df.groupby("rank")["val_rmse"].idxmin()
    ]
    .sort_values("val_rmse")
    .reset_index(drop=True)
)
fine_results_df.to_csv(RESULTS_DIR / "all_new_histories.csv", index=False)
fine_best_by_rank.to_csv(RESULTS_DIR / "new_best_by_rank.csv", index=False)

if fine_failed_configs:
    pd.DataFrame(fine_failed_configs).to_csv(
        RESULTS_DIR / "failed_configs.csv",
        index=False,
    )

print(f"Final fine-search time: {(perf_counter() - fine_search_start) / 60:.1f} min")
display(fine_best_by_rank)

## Compare with the saved rank-16 baseline

In [ ]:
baseline_best_row = baseline_history.loc[
    baseline_history["val_rmse"].idxmin()
]
baseline_best_df = pd.DataFrame([baseline_best_row])

combined_best_by_rank = (
    pd.concat([fine_best_by_rank, baseline_best_df], ignore_index=True)
    .sort_values("val_rmse")
    .reset_index(drop=True)
)
combined_best_by_rank.to_csv(
    RESULTS_DIR / "combined_best_by_rank.csv",
    index=False,
)
display(combined_best_by_rank)

best_row = combined_best_by_rank.iloc[0]
best_hard_rank = int(best_row["rank"])
best_hard_n_iter = int(best_row["iteration"])
best_hard_val_rmse = float(best_row["val_rmse"])

best_parameters_df = pd.DataFrame([{
    "rank": best_hard_rank,
    "n_iter": best_hard_n_iter,
    "val_rmse": best_hard_val_rmse,
}])
best_parameters_df.to_csv(RESULTS_DIR / "best_parameters.csv", index=False)
display(best_parameters_df)

In [ ]:
all_comparison_histories = pd.concat(
    [fine_results_df, baseline_history],
    ignore_index=True,
)

plt.figure(figsize=(9, 5))
for rank in [12, 14, 16, 18]:
    rank_history = all_comparison_histories[
        all_comparison_histories["rank"] == rank
    ].sort_values("iteration")
    plt.plot(
        rank_history["iteration"],
        rank_history["val_rmse"],
        label=f"rank={rank}",
    )

plt.xlabel("Iteration")
plt.ylabel("Validation RMSE")
plt.title("Hard Thresholding: final fine rank comparison")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
best_history = all_comparison_histories[
    all_comparison_histories["rank"] == best_hard_rank
].sort_values("iteration")
best_last_iteration = int(best_history["iteration"].max())

recent_iterations = [250, 270, 280, 290, 300]
display(
    best_history.loc[
        best_history["iteration"].isin(recent_iterations),
        ["iteration", "train_rmse", "val_rmse"],
    ]
)

if best_hard_n_iter == best_last_iteration:
    print("Best iteration is the last one; consider extending.")
else:
    print("Best iteration is inside the trajectory.")